# Create a ***"dat_mnl_dct"***  that has ***"keys"*** = "data_col_lbls" and "***"rows"*** = "dtv"
1. ***"cstm_dat"*** is User defined list of ***"keys"*** from the ***"dat_mnl"***
2. Designed to be used in conjunction with ***"ib_dat_dct"*** to create diverse plotting dctionaries
3. A file named ***"dat_crb"*** is in the WSL folder that holds the xldata tables for different roots. 

## Daily Import of an XL workbook named ***dat_mnl***
1. manually copied from ""https://d.docs.live.net/AED59B3718F319AA/JL_2/dat_crb/dat_mnl.xlsm"
2. to "\wsl.localhost\Ubuntu-20.04\home\ratlabs\JL_2\data\dat_crb" 

## Def functions

In [41]:
# def import_dat_mnl():                                  #Fixed Path  Called Below Returns df_raw, df
import pandas as pd
from pathlib import Path

def import_dat_mnl():
    # Raw import (no header)
    xl_path = Path("/home/ratlabs/JL_2/data/dat_crb/dat_mnl.xlsm")
    df_raw = pd.read_excel(
        xl_path,
        sheet_name="dat_mnl_main",
        header=None,
        engine="openpyxl"
    )

    # Detect the real header row by searching for "col_nms"
    header_row = df_raw.index[df_raw.eq("col_nms").any(axis=1)][0]

    # Promote that row to header
    df = df_raw.copy()
    df.columns = df.iloc[header_row].astype(str)

    # Drop all rows up to and including the header row
    df = df.drop(index=range(header_row + 1)).reset_index(drop=True)

    return df_raw, df



In [42]:
# OLD VERSION def write_df_to_pickle(df, filename):
def write_df_to_pickle(df, filename):
    """
    Writes a DataFrame to a pickle file.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to save.
    filename : str
        The pickle filename, e.g. 'mydata.pkl'.
    """
    df.to_pickle(filename)

# usage 
# write_df_to_pickle(df, "df.pkl")

In [43]:
# OLD VERSION def load_df_from_pickle(filename):

def load_df_from_pickle(filename):
    """
    Loads a DataFrame from a pickle file.

    Parameters
    ----------
    filename : str
        Path to the pickle file.

    Returns
    -------
    pd.DataFrame
    """
    return pd.read_pickle(filename)

    # usage 
    # df = load_df_from_pickle("df.pkl")



In [44]:
def fix_manual_df(df):
    """
    Converts a raw df where:
      - row 0 = column numbers
      - row 1 = actual column names
      - row 2+ = data
    into a clean dataframe with correct headers.
    """

    # Extract row 1 as header
    new_cols = df.iloc[1].tolist()

    # Apply new header
    df_fixed = df.copy()
    df_fixed.columns = new_cols

    # Drop row 0 and row 1
    df_fixed = df_fixed.drop(index=[0, 1]).reset_index(drop=True)

    return df_fixed


In [45]:
def create_plt_lst(dat_col_dct, ib_dct, plt_active_lst):
    """
    Filters dat_col_dct so that:
      - Only columns listed in plt_active_lst are included
      - Each column is filtered to rows whose dtv matches df_77_97_mrn['dtv']

    Returns:
        plt_active_dct : dct
            Keys = column names in plt_active_lst
            Values = filtered pandas Series aligned by dtv
    """

    # Extract the dtv values we want to keep
    dtv_filter_values = set(ib_dct["dtv"].unique())

    plt_active_dct = {}

    for col in plt_active_lst:
        if col not in dat_col_dct:
            continue  # skip missing columns safely

        series = dat_col_dct[col]

        # Filter the series by dtv alignment
        # Assumes dat_col_dct["dtv"] exists and is aligned row‑wise
        dtv_series = dat_col_dct["dtv"]

        filtered_series = series[dtv_series.isin(dtv_filter_values)]

        plt_active_dct[col] = filtered_series.reset_index(drop=True)

    return plt_active_dct


## importing the dat_mnl

## Building dat_mnl_dct

### Update ***"df_dat_mnl"*** from ***"XL"***

In [46]:
df_dat_mnlx,df_dat_mnl= import_dat_mnl()               # Load from XL 

In [47]:
# verify df_dat_mnl   #works

In [48]:
# verify df_dat_mnlx   #works

In [49]:
# verify type(df_dat_mnl)

In [50]:
# verify 
df_dat_mnl.columns.tolist     # Works

<bound method IndexOpsMixin.tolist of Index(['col_nms', 'tst#', 'dtv', 'timestamp', 'Notes', 'urinePH1_5',
       'bullet coffee', 'keto_1', 'Stamina1_5', 'bd_leg_heat',
       ...
       'Column191', 'Column192', 'Column193', 'Column194', 'urine smell',
       'nan', 'nan', 'nan', 'nan', 'nan'],
      dtype='object', name=1, length=231)>

### Edit the ***"cstm_dat_mnl"*** list

In [51]:
df_dat_mnlx = fix_manual_df(df_dat_mnlx)        # Look to 2nd row for column names the numbers are in the first row



In [52]:
# verify df_dat_mnlx

# Creating a plt_lst of a dictionary of dat_mnl cols as the keys that have data in rows "dtv"
1. matches "df_77_97_mrn" "dtv rows"
2. and contains desired "dat_mnl dat_cols"
3. It will be stored in "dat_mnl dct" pkl and used in plot def functions along with  "ib_dct" "dtv rows"

In [53]:
dat_col_dct = {col: df_dat_mnl[col] for col in df_dat_mnl.columns} # Calc the keys to the use for 


In [54]:
# verify 
list(dat_col_dct.keys())             # worked


['col_nms',
 'tst#',
 'dtv',
 'timestamp',
 'Notes',
 'urinePH1_5',
 'bullet coffee',
 'keto_1',
 'Stamina1_5',
 'bd_leg_heat',
 'leg_pn1_5',
 'slp_hr',
 'slp_qlty1_5',
 'slp_wu#',
 'tray a',
 'keto_3',
 'leg_lifts',
 'pull_dwns',
 'pull_ups',
 'pull_ups2',
 'wst_twst',
 'tray b',
 'tray d',
 'blue drnk',
 'bwls_1_5',
 'eggs',
 'eliquist',
 'fast1_5',
 'hot_tub',
 'Urine_Clr',
 'keto_dsrt',
 'keto_2',
 'lazic',
 'MP3oil',
 'nuts',
 'oiled_rd',
 'restruant',
 'rstrnt food',
 'trips',
 'steps 1-5',
 'sun1_5',
 'travel',
 'tray e',
 'VG',
 'vngr-/Bicarb+',
 'wine',
 'mood1_5',
 'mth_blu',
 'Colodal Silver tsp',
 'beer',
 'Omega 3',
 'Slp_LPM',
 'Trk_Rd_PR',
 'Trk_Rd_O2',
 'Trk_Rd_LPM',
 'Trk_Rd_hr',
 'LipoGlud',
 'B12',
 'sgr_avg',
 'sgr_pk',
 'antibiotics',
 'Glycine',
 'creatine',
 'ivermectin',
 'NO',
 'MagGlyc',
 'uritium',
 'wup_mx',
 'auxNAC',
 '8-MTHF',
 'CreGAAtine',
 'eliquist2',
 'Tray a2',
 'CaseinProtein',
 'NnStam',
 'MSNp',
 'Nn_Mood',
 'NghtBag  Cups 10oz',
 'Trk Pr_0 LPM0?

In [55]:
# verify dat_col_dct    #worked

In [56]:
dtv = dat_col_dct["dtv"]
# verify   dtv  # worked has all days

In [57]:
notes = dat_col_dct["Notes"]

In [58]:
# verify notes           #works

## Write the ***"dat_col_dct"*** to Pickle so it can be used to create plot by going down dictionaries

In [59]:
import pickle
with open("df_77_97_mrn.pkl", "rb") as f:  
    df_77_97_mrn = pickle.load(f)

In [60]:
# verify df_77_97_mrn #Works

In [61]:
plt_active_lst = ["dtv", "timestamp", "Notes"]  # 


In [62]:
plt_active_lst


['dtv', 'timestamp', 'Notes']

In [63]:
print(df_77_97_mrn.columns.tolist())


['timestamp', 'dtv', 'weight', 'vfa_(visceral_fat_area)', 'ecw/tbw', 'ecw/tbw_of_left_leg_x', 'ecw/tbw_of_right_leg_x', 'bmr_(basal_metabolic_rate)', 'smm_(skeletal_muscle_mass)', 'khz-whole_body_phase_angle', 'whole_body_ecw/tbw_t_score', 'ecw_(extracellular_water)', 'icw_(intracellular_water)', 'ecw/tbw_of_left_leg_y', 'ecw/tbw_of_right_leg_y', 'ecw_of_left_leg', 'ecw_of_right_leg', 'lower_limit_(ecw_of_left_leg_normal_range)', 'lower_limit_(ecw_of_right_leg_normal_range)', 'upper_limit_(ecw_of_left_leg_normal_range)', 'upper_limit_(ecw_of_right_leg_normal_range)']


In [64]:
plt_active_dct = (dat_col_dct, df_77_97_mrn, plt_active_lst)

In [65]:
# verify plt_active_dct["dtv"]                    # worked
# verify plt_active_dct["timestamp"]             # worked
# verify plt_active_dct["slp_hr"]  

In [66]:
for i, col in enumerate(df_77_97_mrn.columns):
    print(i, repr(col))


0 'timestamp'
1 'dtv'
2 'weight'
3 'vfa_(visceral_fat_area)'
4 'ecw/tbw'
5 'ecw/tbw_of_left_leg_x'
6 'ecw/tbw_of_right_leg_x'
7 'bmr_(basal_metabolic_rate)'
8 'smm_(skeletal_muscle_mass)'
9 'khz-whole_body_phase_angle'
10 'whole_body_ecw/tbw_t_score'
11 'ecw_(extracellular_water)'
12 'icw_(intracellular_water)'
13 'ecw/tbw_of_left_leg_y'
14 'ecw/tbw_of_right_leg_y'
15 'ecw_of_left_leg'
16 'ecw_of_right_leg'
17 'lower_limit_(ecw_of_left_leg_normal_range)'
18 'lower_limit_(ecw_of_right_leg_normal_range)'
19 'upper_limit_(ecw_of_left_leg_normal_range)'
20 'upper_limit_(ecw_of_right_leg_normal_range)'


In [67]:
# This is the ready to plot list of combined 97 77 data of most interest
import pickle

write_df_to_pickle(df_77_97_mrn, "df_77_97_mrn.pkl")
print("df_77_97_mrn written to pickle")
# verify df_77_97_mrn

df_77_97_mrn written to pickle
